<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-3-prompt-engineering/week-3/week3_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Prompts as Engineering Artifacts

Task: classify a code review comment into bug, security, performance, or style, with a one-sentence rationale. Runs fixture-based: no API key required. Set `GEMINI_API_KEY` to run the prompts for real instead.

Dependencies: `sentence-transformers` (local; needs internet the first time, to download the embedding model).

In [1]:
import os

BRANCH = "week-3-prompt-engineering"
REPO_URL = "https://github.com/BenMillerDev/Applied-LLM-Systems.git"

if not os.path.exists('prompts'):
    # prompts/ isn't sitting next to this notebook already (e.g. it was opened
    # on its own rather than as part of a full repo clone) -- pull it down
    !rm -rf _repo
    !git clone -b {BRANCH} {REPO_URL} _repo
    !cp -r _repo/week-3/prompts .

In [2]:
import json, pathlib
def gemini_chat(messages, model='gemini-2.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: False (fixtures used when False)


In [3]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite

I chose code review triage: classify a review comment into bug, security, performance, or style. My team uses a Claude-based code-reviewer skill at work that does something similar, so this is close to a real task rather than a made-up one.

Prompt versions live under `prompts/code_review_v1.md` and `prompts/code_review_v2.md`: a system message, four few-shot examples (one per category, none of them reused from the test set below), and chain-of-thought scaffolding that asks for a short, visible rationale in the JSON output rather than hidden reasoning. Loaded from disk here so the notebook still runs standalone.

v2 adds one rule after the few-shot examples: when a comment raises more than one concern, classify by the most severe applicable category (security > bug > performance > style) rather than whichever issue is mentioned first.

In [4]:
PROMPTS_DIR = pathlib.Path('prompts')

def load_prompt(version):
    # prompt files are the actual versioned artifact -- read from disk the
    # same way any other file under version control would be
    path = PROMPTS_DIR / f'code_review_{version}.md'
    return path.read_text()

PROMPT_V1 = load_prompt('v1')
PROMPT_V2 = load_prompt('v2')

print(f'v1 prompt: {len(PROMPT_V1)} chars')
print(f'v2 prompt: {len(PROMPT_V2)} chars (adds the severity-priority rule after the few-shot examples)')
print()
print('--- v1, first 400 chars ---')
print(PROMPT_V1[:400])

v1 prompt: 1541 chars
v2 prompt: 2225 chars (adds the severity-priority rule after the few-shot examples)

--- v1, first 400 chars ---
# Code Review Triage Prompt — v1

## System

You are a code review triage assistant. Read each review comment and
classify the issue it raises into exactly one of these categories: bug,
security, performance, style.

Before answering, briefly identify the core issue the comment is raising.
Then respond with a JSON object with two fields:

- `category`: one of bug, security, performance, style
- `r


## Part 2: Build the test suite

12 cases, three for each category (bug, security, performance, style), each with an expected `category` and a short expected `rationale`. Every response is scored two ways: an exact-match check on `category`, and a semantic-similarity score comparing the model's `rationale` to the expected one via `sentence-transformers`, since two rationales can describe the same issue in different words.

In [5]:
tests = [
  {'id':1,'comment':'This loop uses `<` instead of `<=` when checking the array bound, so the last item never gets processed.','category':'bug','rationale':'off-by-one error skips the last array element'},
  {'id':2,'comment':'The catch block here swallows the exception without logging it, so failures are invisible in production.','category':'bug','rationale':'exception is silently swallowed with no logging'},
  {'id':3,'comment':"We're concatenating raw user input directly into the SQL query string here.",'category':'security','rationale':'raw user input concatenated into a SQL query, injection risk'},
  {'id':4,'comment':"This endpoint doesn't check if the requesting user actually owns the resource before returning it.",'category':'security','rationale':'missing authorization check, broken access control'},
  {'id':5,'comment':'This does a database call inside the loop, we should batch this into a single query.','category':'performance','rationale':'N+1 query pattern, one DB call per loop iteration'},
  {'id':6,'comment':"We're recalculating this value on every render instead of memoizing it.",'category':'performance','rationale':'redundant recomputation instead of caching or memoizing'},
  {'id':7,'comment':"Let's rename `x` and `y` to something more descriptive, like `userId` and `orderId`.",'category':'style','rationale':"variable names aren't descriptive"},
  {'id':8,'comment':'Minor, but this function could use a docstring explaining what it returns.','category':'style','rationale':'missing docstring, not a functional issue'},
  {'id':9,'comment':'`tmp2` is a confusing name, and also the loop bound should probably be `<=` not `<`, otherwise we skip the last element.','category':'bug','rationale':'off-by-one bug is the substantive issue, the naming is a secondary nit'},
  {'id':10,'comment':"This function is called `validateUserSecurity`, but it only checks that the username field isn't empty, nothing security-related. Might be worth a less misleading name.",'category':'style','rationale':'renaming a misleadingly-named function for clarity, not an actual vulnerability'},
  {'id':11,'comment':"This endpoint takes a redirect URL from the query string and sends the user there without checking it's actually one of our own domains.",'category':'security','rationale':'unvalidated redirect target allows open-redirect attacks'},
  {'id':12,'comment':'This function pulls an entire 10 million row table into memory just to count how many rows match a condition.','category':'performance','rationale':'loads the full dataset into memory instead of using a database count query'},
]
print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 12


In [6]:
# Labeled fixtures stand in for model output when LIVE is False. v2 fixes #9 but regresses #10.
FIX = {
  'v1': {1:('bug','loop bound is off by one, skips last element'),
         2:('bug','exception is caught and discarded silently'),
         3:('security','SQL injection risk from unsanitized input'),
         4:('security','missing ownership check before returning data'),
         5:('performance','N+1 query pattern from calling the DB in a loop'),
         6:('performance','value is recomputed every render instead of cached'),
         7:('style','variable names are not descriptive'),
         8:('style','function is missing a docstring'),
         9:('style','variable name tmp2 is unclear'),
         10:('style','function name is misleading given what it actually checks'),
         11: ('security', 'unvalidated redirect target allows open-redirect'),
         12: ('performance', 'loads the full table into memory instead of counting in the database')},
  'v2': {1:('bug','loop bound is off by one, skips last element'),
         2:('bug','exception is caught and discarded silently'),
         3:('security','SQL injection risk from unsanitized input'),
         4:('security','missing ownership check before returning data'),
         5:('performance','N+1 query pattern from calling the DB in a loop'),
         6:('performance','value is recomputed every render instead of cached'),
         7:('style','variable names are not descriptive'),
         8:('style','function is missing a docstring'),
         9:('bug','loop bound is off by one; naming is a secondary issue'),
         10:('security','flagged due to security-related naming and validation context'),
         11: ('security', 'unvalidated redirect target allows open-redirect'),
         12: ('performance', 'loads the full table into memory instead of counting in the database')},
}

def build_messages(prompt_template, comment_text):
    # fills the {comment} placeholder in the versioned prompt file with the
    # actual review comment for this test case
    filled = prompt_template.replace('{comment}', comment_text)
    return [{'role': 'user', 'content': filled}]

def run_case(version, prompt_template, test_case):
    if LIVE:
        response_text = gemini_chat(build_messages(prompt_template, test_case['comment']))
        try:
            parsed_response = json.loads(response_text)
            return parsed_response.get('category', ''), parsed_response.get('rationale', '')
        except Exception:
            return '', response_text or ''
    return FIX[version][test_case['id']]

def score(version, prompt_template):
    scored_rows = []
    for test_case in tests:
        predicted_category, predicted_rationale = run_case(version, prompt_template, test_case)
        scored_rows.append({
            'id': test_case['id'],
            'exact': exact_match(test_case['category'], predicted_category),
            'sem': semantic_sim(test_case['rationale'], predicted_rationale),
            'got': predicted_category,
        })
    exact_match_accuracy = sum(row['exact'] for row in scored_rows) / len(scored_rows)
    return exact_match_accuracy, scored_rows

v1_accuracy, r1 = score('v1', PROMPT_V1)
v2_accuracy, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {v1_accuracy:.0%}   v2 exact-match {v2_accuracy:.0%}')

v1 exact-match 92%   v2 exact-match 92%


## Part 3 and 4: the tradeoff and the failure


In [7]:
def average_semantic_sim(rows):
    # mean semantic-similarity score across all cases for one prompt version
    return sum(row['sem'] for row in rows) / len(rows)

def find_row(rows, case_id):
    # looks up one test case's scored result by id
    return next(row for row in rows if row['id'] == case_id)

print('Average semantic-similarity:')
print(f'  v1: {average_semantic_sim(r1):.3f}')
print(f'  v2: {average_semantic_sim(r2):.3f}')
print()

for t in tests:
    row1 = find_row(r1, t['id'])
    row2 = find_row(r2, t['id'])
    if row1['exact'] != row2['exact']:
        verdict = 'IMPROVED' if row2['exact'] > row1['exact'] else 'REGRESSED'
        print(f"#{t['id']} expected {t['category']!r}  [{verdict}]")
        print(f"  v1 -> {row1['got']!r}  (exact={row1['exact']}, sem={row1['sem']:.3f})")
        print(f"  v2 -> {row2['got']!r}  (exact={row2['exact']}, sem={row2['sem']:.3f})")

Average semantic-similarity:
  v1: 0.721
  v2: 0.695

#9 expected 'bug'  [IMPROVED]
  v1 -> 'style'  (exact=0.0, sem=0.273)
  v2 -> 'bug'  (exact=1.0, sem=0.331)
#10 expected 'style'  [REGRESSED]
  v1 -> 'style'  (exact=1.0, sem=0.757)
  v2 -> 'security'  (exact=0.0, sem=0.388)


## Comparison table to view results

In [8]:
import pandas as pd

def build_comparison_table(test_cases, v1_rows, v2_rows):
    # one row per test case, showing what each prompt version predicted
    # side by side so the full run is visible, not just the cases that changed
    table_rows = []
    for test_case in test_cases:
        v1_row = find_row(v1_rows, test_case['id'])
        v2_row = find_row(v2_rows, test_case['id'])
        v1_is_correct = v1_row['exact'] == 1.0
        v2_is_correct = v2_row['exact'] == 1.0
        if v1_is_correct == v2_is_correct:
            verdict = 'consistent'
        elif v2_is_correct:
            verdict = 'IMPROVED'
        else:
            verdict = 'REGRESSED'
        table_rows.append({
            'id': test_case['id'],
            'expected': test_case['category'],
            'v1 predicted': v1_row['got'],
            'v1 correct': '✓' if v1_is_correct else '✗',
            'v1 similarity': v1_row['sem'],
            'v2 predicted': v2_row['got'],
            'v2 correct': '✓' if v2_is_correct else '✗',
            'v2 similarity': v2_row['sem'],
            'verdict': verdict,
        })
    return pd.DataFrame(table_rows).set_index('id')

def highlight_changed_rows(row):
    # flags the cases where the edit changed the outcome so the tradeoff
    # is visible at a glance instead of requiring a manual side-by-side read
    if row['verdict'] == 'consistent':
        return [''] * len(row)
    color = 'lightgreen' if row['verdict'] == 'IMPROVED' else 'lightcoral'
    return [f'background-color: {color}'] * len(row)

comparison_table = build_comparison_table(tests, r1, r2)
comparison_table.style.apply(highlight_changed_rows, axis=1).format({'v1 similarity': '{:.3f}', 'v2 similarity': '{:.3f}'})

,expected,v1 predicted,v1 correct,v1 similarity,v2 predicted,v2 correct,v2 similarity,verdict
id,,,,,,,,
1,bug,bug,✓,0.608,bug,✓,0.608,consistent
2,bug,bug,✓,0.746,bug,✓,0.746,consistent
3,security,security,✓,0.775,security,✓,0.775,consistent
4,security,security,✓,0.504,security,✓,0.504,consistent
5,performance,performance,✓,0.939,performance,✓,0.939,consistent
6,performance,performance,✓,0.487,performance,✓,0.487,consistent
7,style,style,✓,0.988,style,✓,0.988,consistent
8,style,style,✓,0.875,style,✓,0.875,consistent
9,bug,style,✗,0.273,bug,✓,0.331,IMPROVED


### Results

v1 and v2 both land at 92% exact-match (11 of 12), but they miss different cases: v1 misses #9 (called it style, should be bug), v2 fixes #9 but misses #10 (called it security, should be style). For case #9, the semantic similarity only went from 0.273 to 0.331, so it's not a big jump, but the classification itself did get fixed in v2. Case #10 went the other way. Its semantic similarity dropped from 0.757 down to 0.388, and the classification flipped from correct to incorrectly picking security instead of style.

So the v2 edit didn't raise accuracy overall.  It traded one wrong case for a different wrong case.

### Cause

The v2 rule tells the model to classify by the most severe category whenever a comment raises more than one concern, or uses language associated with a more severe category. For #9, that works the way I wanted it to. It correctly promotes the buried off-by-one bug over the naming nit that gets mentioned first. For #10 though, there isn't actually a second concern in the comment. It's just the word "security" sitting inside a function name that the comment is criticizing for being misleading. The rule can't tell the difference between a comment that's actually about a security issue and a comment that just happens to contain the word security while talking about something else entirely. It's reacting to the vocabulary instead of what the comment is actually reporting.

I think that's the same failure as the #9 problem, just flipped. I built the rule to look past whatever is mentioned first and find the real severity, and it ended up finding severity that isn't there, because it's pattern-matching on words instead of on what the comment is actually claiming.

### Mitigation

Reverting the rule just brings back the #9 miss, that's trading errors back and forth rather than resolving anything. A more targeted fix is to make the rule's condition about substance, not vocabulary: something like "only classify by a more severe category if the comment describes a concrete instance of that category's problem, not merely because a related word appears." Pairing that with a fifth few-shot example, a comment that uses risk-adjacent language without describing an actual vulnerability, would give the model a concrete pattern to match against instead of a rule that can be satisfied by keyword presence alone.

## Part 5: Submit
Run top to bottom on your prompt, then open a pull request with a result summary, the notebook, and a linked research-note issue. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).